# 06 — Análise de Viés por Subgrupo

Avalia a performance do modelo `mlp_8010_ohe_b16` separadamente para
subgrupos demográficos do **blind test set** (n=705, split 80/10/10, seed=42).

Os resultados aqui produzidos alimentam diretamente a **seção 6 do MODEL_CARD.md**.

Subgrupos analisados:
- `Gender`: Male vs. Female
- `Senior Citizen`: Sim vs. Não
- Faixas de `Tenure Months`: 0–12 / 13–36 / 37–60 / 61+

Métricas: **ROC-AUC**, **F1-score** e **FNR** (False Negative Rate = taxa de churners não detectados).

## 1. Setup

In [1]:
from __future__ import annotations

import sys
import tempfile
from pathlib import Path

import joblib
import mlflow
import mlflow.artifacts
import mlflow.pytorch
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import confusion_matrix, f1_score, roc_auc_score

ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT / "src"))

from churn.config import DEPLOY_THRESHOLD, MLFLOW_EXPERIMENT_NAME, MLFLOW_RUN_NAME, SEED
from churn.data.loader import load_raw_data
from churn.data.preprocessing import clean_raw, split_features_target, stratified_split

## 2. Dados — Blind Test Set

In [2]:
df_raw = load_raw_data()
df_clean = clean_raw(df_raw)
X, y = split_features_target(df_clean)
splits = stratified_split(X, y, test_size=0.10, val_size=0.10, seed=SEED)

X_test: pd.DataFrame = splits.X_test
y_test: pd.Series = splits.y_test

print(f"Test set: {len(X_test)} samples | {y_test.sum()} churners ({y_test.mean():.1%})")

Test set: 705 samples | 187 churners (26.5%)


## 3. Carregamento do Modelo (MLflow)

In [3]:
mlflow.set_tracking_uri((ROOT / "mlruns").as_uri())
runs = mlflow.search_runs(
    experiment_names=[MLFLOW_EXPERIMENT_NAME],
    filter_string=f"tags.mlflow.runName = '{MLFLOW_RUN_NAME}'",
    order_by=["start_time DESC"],
)
assert not runs.empty, f"Run '{MLFLOW_RUN_NAME}' não encontrado — re-execute 04_mlp.ipynb."

run_id = runs.iloc[0]["run_id"]
print(f"run_id: {run_id}")

model = mlflow.pytorch.load_model(f"runs:/{run_id}/mlp_model")
model.eval()

with tempfile.TemporaryDirectory() as tmpdir:
    pp_path = mlflow.artifacts.download_artifacts(
        artifact_uri=f"runs:/{run_id}/preprocessor.joblib",
        dst_path=tmpdir,
    )
    preprocessor = joblib.load(pp_path)

print("Modelo e preprocessador carregados.")

run_id: 24dc087969f0430fadc011e1ce5208a1
Modelo e preprocessador carregados.


## 4. Inferência no Test Set

In [4]:
X_proc = preprocessor.transform(X_test).astype(np.float32)

with torch.no_grad():
    logits = model(torch.tensor(X_proc))
    probs: np.ndarray = torch.sigmoid(logits).squeeze().numpy()

preds = (probs >= DEPLOY_THRESHOLD).astype(int)

print(f"Overall ROC-AUC: {roc_auc_score(y_test, probs):.4f}  (esperado ~0.865)")

Overall ROC-AUC: 0.8651  (esperado ~0.865)


## 5. Análise por Subgrupo

In [5]:
def compute_metrics(y_true: np.ndarray, y_prob: np.ndarray, y_pred: np.ndarray) -> dict:
    n_pos = int(y_true.sum())
    if n_pos == 0 or len(np.unique(y_true)) < 2:
        return {"n": len(y_true), "n_churners": n_pos, "auc": float("nan"), "f1": float("nan"), "fnr": float("nan")}
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        "n": len(y_true),
        "n_churners": n_pos,
        "auc": roc_auc_score(y_true, y_prob),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "fnr": fn / (fn + tp) if (fn + tp) > 0 else 0.0,
    }

In [6]:
rows = []

rows.append({"Subgrupo": "Overall", **compute_metrics(y_test.values, probs, preds)})

# Gender
for g in ["Male", "Female"]:
    mask = X_test["Gender"] == g
    rows.append({"Subgrupo": f"Gender — {g}", **compute_metrics(y_test[mask].values, probs[mask], preds[mask])})

# Senior Citizen (raw data: 0/1; após clean_raw: 'No'/'Yes')
for val, label in [("No", "Não"), ("Yes", "Sim")]:
    mask = X_test["Senior Citizen"] == val
    rows.append({"Subgrupo": f"Senior Citizen — {label}", **compute_metrics(y_test[mask].values, probs[mask], preds[mask])})

# Tenure bins
for label, lo, hi in [("0–12m", 0, 12), ("13–36m", 13, 36), ("37–60m", 37, 60), ("61+m", 61, 9999)]:
    mask = (X_test["Tenure Months"] >= lo) & (X_test["Tenure Months"] <= hi)
    rows.append({"Subgrupo": f"Tenure {label}", **compute_metrics(y_test[mask].values, probs[mask], preds[mask])})

df_results = pd.DataFrame(rows).rename(columns={"n": "N", "n_churners": "Churners", "auc": "ROC-AUC", "f1": "F1", "fnr": "FNR"})
df_results[["ROC-AUC", "F1", "FNR"]] = df_results[["ROC-AUC", "F1", "FNR"]].round(4)
display(df_results)

,Subgrupo,N,Churners,ROC-AUC,F1,FNR
0,Overall,705,187,0.8651,0.6073,0.0695
1,Gender — Male,352,95,0.8811,0.6312,0.0632
2,Gender — Female,353,92,0.8501,0.5842,0.0761
3,Senior Citizen — Não,591,133,0.8671,0.5674,0.0977
4,Senior Citizen — Sim,114,54,0.7775,0.7200,0.0000
5,Tenure 0–12m,206,100,0.7775,0.6899,0.0100
6,Tenure 13–36m,186,52,0.8652,0.6065,0.0962
7,Tenure 37–60m,159,23,0.7708,0.4086,0.1739
8,Tenure 61+m,154,12,0.9366,0.4737,0.2500


## 6. Interpretação

### Gênero (Δ AUC = 0,031)
O modelo discrimina melhor em clientes masculinos (0,881) do que femininos (0,850). O gap é moderado e reflete correlações indiretas com features de serviço e contrato — `Gender` tem Cramér's V ≈ 0,008 com o target (praticamente ruído).

### Senior Citizen (Δ AUC = 0,089 — gap mais relevante)
- **AUC 0,778** para sênior vs. 0,867 para não-sênior — diferença expressiva.
- **FNR = 0,000** para sênior parece positivo mas é artefato: com threshold baixo (0,27) e taxa de churn de 47% neste grupo, o modelo classifica quase todos como churn, inflando FP.
- **Ação sugerida:** monitorar FP em sênior em produção; avaliar threshold diferenciado em versões futuras.

### Faixas de Tenure — padrão não-linear
| Faixa | Insight |
|---|---|
| 0–12m | AUC baixo (0,778) mas FNR quase zero — modelo agressivo, muitos FP, adequado dado alto custo de FN |
| 13–36m | Performance próxima à média global |
| 37–60m | FNR em ascensão (0,174) — clientes maduros que decidem cancelar são mais difíceis de prever |
| 61+m | AUC mais alto (0,937) mas FNR = 0,250 — poucos churners (n=12), difíceis de capturar |

**Segmentos prioritários para monitoramento pós-deploy:** Senior Citizen (AUC 0,778) e Tenure 37–60m (FNR 17%).